# 🔬 Gender Persona Bias — Visualization Notebook
**Research:** Persona-Induced Bias in LLMs | Domain: Gender | Benchmarks: GSM8K + ETHICS  
**Models:** 6 Main Models (Thinking vs Instruct, Small/Medium/Large, Dense/MoE)  
**Analyses:** Accuracy · P-Value (Z-Test) · Cohen's h · Delta Analysis · Token Usage · Linguistic Marker

---
> **How to use:** Run cells top-to-bottom. Modify `CONFIG` section to change colors, model order, or figure sizes.


## 0. Setup & Installation

In [1]:
# Install required packages
!pip install gspread gspread-dataframe scipy seaborn matplotlib --quiet



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Google Sheets Authentication & Connection

In [9]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current directory:
d:\TugasAkhir\src\notebook

Files:
['aggregated_3_groups_markers.csv', 'aggregated_5_groups_markers.csv', 'comprehensive_linguistic_bias_report.csv', 'CoT_Discourse_Marker_Analysis_v3.ipynb', 'CoT_linguistic_marker.ipynb', 'CoT_linguistic_marker_GSM8K.ipynb', 'CoT_linguistic_marker_GSM8K_final.ipynb', 'CoT_linguistic_marker_GSM8K_revised.ipynb', 'detailed_bias_analysis_summary.csv', 'figures_gender', 'Gender_Bias_Visualization.ipynb', 'GSM8K_persona_man_&_woman.ipynb', 'image.png', 'linguistic_marker_ETHICS_v9 (1).ipynb', 'linguistic_marker_religion_persona.ipynb', 'persona_man_&_woman.ipynb', 'run_radar_analysis.py', 'TA - Accuracy Woman & Man Persona - Model Accuracy [GSM8K].csv', 'token_usage_3_groups.csv', 'token_usage_5_groups.csv']


In [10]:
from pathlib import Path

credential_path = Path("../../credentials.json")

print(credential_path.resolve())

D:\TugasAkhir\credentials.json


In [11]:
import gspread
from google.oauth2.service_account import Credentials
from pathlib import Path

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

credential_path = Path("../../credentials.json")

creds = Credentials.from_service_account_file(
    credential_path,
    scopes=SCOPES
)

gc = gspread.authorize(creds)

sh = gc.open_by_key(
    "17cSsi5b1aMQFjjc7PHKgz0oeBlnQ9jiNXKdYWJ4Gjqk"
)

print("✅ Connected:", sh.title)

FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\credentials.json'

In [14]:
import gspread
from google.auth import default
import pandas as pd
import numpy as np

creds, _ = default()
gc = gspread.authorize(creds)

SHEET_ID = '17cSsi5b1aMQFjjc7PHKgz0oeBlnQ9jiNXKdYWJ4Gjqk'
sh = gc.open_by_key(SHEET_ID)
print("✅ Connected to:", sh.title)
print("Available sheets:", [ws.title for ws in sh.worksheets()])


DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

## 2. Configuration — Edit Here

In [2]:
# ──────────────────────────────────────────────────────────────────────────
# MODEL MAPPING: raw Google Sheets name → display label
# Edit if your sheet uses different names
# ──────────────────────────────────────────────────────────────────────────
MODEL_MAP_GSM8K = {
    'deepseek-r1-distill-qwen-32b':  'DeepSeek R1 32B',
    'Qwen3 Next 80B A3B Thinking':   'Qwen3-Think 80B',
    'Meta Llama 3.3 70B Instruct':   'LLaMA 3.3 70B',
    'OpenAI GPT-OSS-20B':            'GPT-OSS 20B',
    'Meta Llama 3.1 8B Instruct':    'LLaMA 3.1 8B',
    'Mistral Ministral 8B':          'Ministral 8B',
}

MODEL_MAP_ETHICS = {
    'deepseek_deepseek-r1-distill-qwen-32b': 'DeepSeek R1 32B',
    'qwen_qwen3-next-80b-a3b-thinking':      'Qwen3-Think 80B',
    'meta-llama/llama-3.3-70b-instruct':     'LLaMA 3.3 70B',
    'openai_gpt-oss-20b':                    'GPT-OSS 20B',
    'meta-llama_llama-3.1-8b-instruct':      'LLaMA 3.1 8B',
    'mistralai_ministral-8b-2410':           'Ministral 8B',
}

# Display order (left to right)
MODEL_ORDER = [
    'DeepSeek R1 32B',
    'Qwen3-Think 80B',
    'LLaMA 3.3 70B',
    'GPT-OSS 20B',
    'LLaMA 3.1 8B',
    'Ministral 8B',
]

MODEL_TYPE = {
    'DeepSeek R1 32B': 'Thinking',
    'Qwen3-Think 80B': 'Thinking',
    'LLaMA 3.3 70B':   'Instruct',
    'GPT-OSS 20B':     'Instruct',
    'LLaMA 3.1 8B':    'Instruct',
    'Ministral 8B':    'Instruct',
}

# Color palette
C = {
    'man_exp':  '#2166AC',
    'man_imp':  '#74ADD1',
    'w_exp':    '#D6604D',
    'w_imp':    '#F4A582',
    'neutral':  '#777777',
    'thinking': '#2166AC',
    'instruct': '#888888',
}

# Figure output (None = display inline, 'png' = also save)
SAVE_FIGS = True
SAVE_DIR  = 'figures_gender'

import os
if SAVE_FIGS:
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"✅ Figures will be saved to ./{SAVE_DIR}/")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import norm
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'figure.dpi': 150,
})

def savefig(name):
    if SAVE_FIGS:
        path = f'{SAVE_DIR}/{name}.png'
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  💾 Saved: {path}")
    plt.show()

print("✅ Config loaded")


✅ Figures will be saved to ./figures_gender/
✅ Config loaded


## 3. Load & Parse Data from Google Sheets

In [3]:

def parse_sheet(worksheet, model_map):
    """
    Parse multi-block sheet format:
    - Each model has a header row in col B (e.g. 'Model: deepseek-r1-...')
    - Followed by column headers row
    - Followed by data rows
    - Ends with ALL/ALL summary row
    Returns: dict {display_name: {persona: {correct, total, accuracy, prompt_tokens, completion_tokens}}}
    """
    raw = worksheet.get_all_values()
    result = {}
    current_model = None
    col_map = {}  # column name → index

    for row in raw:
        if not any(row): continue

        # Model header: col B has 'Model: ...' pattern
        if len(row) >= 2 and row[1].startswith('Model:'):
            raw_name = row[1].replace('Model:', '').strip()
            if raw_name in model_map:
                current_model = model_map[raw_name]
                result[current_model] = {}
                col_map = {}
            else:
                current_model = None
            continue

        # Col A single-model header (some sheets use col A)
        if row[0] and not row[1] and current_model is None:
            for raw_name, label in model_map.items():
                if raw_name in row[0]:
                    current_model = label
                    result[current_model] = {}
                    col_map = {}
                    break
            continue

        if current_model is None:
            continue

        # Column header row
        if 'persona_folder' in row or 'persona_name' in row:
            col_map = {v: i for i, v in enumerate(row) if v}
            continue

        # Data row
        if not col_map: continue
        persona_col = col_map.get('persona_folder') or col_map.get('persona_name')
        if persona_col is None: continue

        persona = row[persona_col] if len(row) > persona_col else ''
        if not persona or persona == 'ALL': continue

        try:
            acc_col   = col_map.get('accuracy') or col_map.get('correct_accuracy')
            corr_col  = col_map.get('correct')
            tot_col   = col_map.get('total')
            ptok_col  = col_map.get('sum_prompt_tokens')
            ctok_col  = col_map.get('sum_completion_tokens')

            acc   = float(row[acc_col])  if (acc_col  is not None and len(row) > acc_col  and row[acc_col])  else None
            corr  = float(row[corr_col]) if (corr_col is not None and len(row) > corr_col and row[corr_col]) else None
            tot   = float(row[tot_col])  if (tot_col  is not None and len(row) > tot_col  and row[tot_col])  else None
            ptok  = float(row[ptok_col]) if (ptok_col is not None and len(row) > ptok_col and row[ptok_col]) else None
            ctok  = float(row[ctok_col]) if (ctok_col is not None and len(row) > ctok_col and row[ctok_col]) else None

            if acc is not None:
                result[current_model][persona] = {
                    'accuracy': acc,
                    'correct':  corr,
                    'total':    tot,
                    'sum_prompt_tokens':     ptok,
                    'sum_completion_tokens': ctok,
                    'avg_prompt_tokens':     ptok/tot if (ptok and tot) else None,
                    'avg_completion_tokens': ctok/tot if (ctok and tot) else None,
                }
        except (ValueError, IndexError, TypeError):
            continue

    return result

# Load sheets
ws_gsm8k  = sh.worksheet('Gender Persona GSM8K')
ws_ethics = sh.worksheet('Gender Persona ETHICS')

gsm8k  = parse_sheet(ws_gsm8k,  MODEL_MAP_GSM8K)
ethics = parse_sheet(ws_ethics, MODEL_MAP_ETHICS)

print("✅ GSM8K loaded — models:", list(gsm8k.keys()))
print("✅ ETHICS loaded — models:", list(ethics.keys()))

# Quick sanity check
for m in MODEL_ORDER:
    gsm_ok  = '✅' if m in gsm8k  else '❌'
    eth_ok  = '✅' if m in ethics else '❌'
    print(f"  {gsm_ok} GSM8K  |  {eth_ok} ETHICS  — {m}")


NameError: name 'sh' is not defined

## 4. Statistical Computations

In [4]:
PERSONAS_MAN  = ['man_implicit_1','man_implicit_2','man_implicit_3']
PERSONAS_WMAN = ['woman_implicit_1','woman_implicit_2','woman_implicit_3']
PERSONAS_ALL  = ['man_explicit_1'] + PERSONAS_MAN + ['woman_explicit_1'] + PERSONAS_WMAN + ['neutral']

COMPARISONS = [
    ('M_avg vs Neutral', ['man_explicit_1'] + PERSONAS_MAN,   ['neutral']),
    ('W_avg vs Neutral', ['woman_explicit_1'] + PERSONAS_WMAN, ['neutral']),
    ('M_exp vs Imp',     ['man_explicit_1'],                   PERSONAS_MAN),
    ('W_exp vs Imp',     ['woman_explicit_1'],                 PERSONAS_WMAN),
    ('M_imp vs Neutral', PERSONAS_MAN,                        ['neutral']),
    ('W_imp vs Neutral', PERSONAS_WMAN,                       ['neutral']),
]
COMP_LABELS = [c[0] for c in COMPARISONS]

def pool(model_data, personas):
    c = sum(model_data[p]['correct'] for p in personas
            if p in model_data and model_data[p].get('correct'))
    t = sum(model_data[p]['total']   for p in personas
            if p in model_data and model_data[p].get('total'))
    return c, t

def z_test_pval(c1, t1, c2, t2):
    if not (t1 and t2): return np.nan
    p1, p2  = c1/t1, c2/t2
    p_pool  = (c1+c2)/(t1+t2)
    se      = np.sqrt(p_pool*(1-p_pool)*(1/t1+1/t2))
    if se == 0: return np.nan
    return 2*(1 - norm.cdf(abs((p1-p2)/se)))

def cohen_h(p1, p2):
    return 2*(np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))

def avg_acc(model_data, personas):
    vals = [model_data[p]['accuracy'] for p in personas if p in model_data]
    return np.nanmean(vals) if vals else np.nan

def compute_stat_matrices(dataset):
    pvals = {}; hs = {}
    for m in MODEL_ORDER:
        if m not in dataset: continue
        d = dataset[m]
        pvals[m] = {}; hs[m] = {}
        for label, g1, g2 in COMPARISONS:
            c1,t1 = pool(d,g1); c2,t2 = pool(d,g2)
            p = z_test_pval(c1,t1,c2,t2)
            pvals[m][label] = p
            if t1 and t2:
                hs[m][label] = cohen_h(c1/t1, c2/t2)
    return pvals, hs

def compute_deltas(dataset):
    out = {m: {} for m in MODEL_ORDER}
    for m in MODEL_ORDER:
        if m not in dataset: continue
        d = dataset[m]
        if 'neutral' not in d: continue
        neut  = d['neutral']['accuracy']
        m_exp = d.get('man_explicit_1',   {}).get('accuracy', np.nan)
        w_exp = d.get('woman_explicit_1', {}).get('accuracy', np.nan)
        m_imp = avg_acc(d, PERSONAS_MAN)
        w_imp = avg_acc(d, PERSONAS_WMAN)
        out[m] = {
            'delta_man_neutral':   np.nanmean([m_exp, m_imp]) - neut,
            'delta_woman_neutral': np.nanmean([w_exp, w_imp]) - neut,
            'delta_man_ei':        m_imp - m_exp,
            'delta_woman_ei':      w_imp - w_exp,
            'gender_gap':          abs(np.nanmean([m_exp,m_imp]) - np.nanmean([w_exp,w_imp])),
            'ei_gap':              (m_imp+w_imp)/2 - (m_exp+w_exp)/2,
            'neutral':             neut,
        }
    return out

gsm_pv, gsm_h  = compute_stat_matrices(gsm8k)
eth_pv, eth_h  = compute_stat_matrices(ethics)
gsm_delta = compute_deltas(gsm8k)
eth_delta = compute_deltas(ethics)

print("✅ Statistics computed")
print("\nSample — DeepSeek R1 32B GSM8K deltas:")
for k, v in gsm_delta.get('DeepSeek R1 32B', {}).items():
    print(f"  {k}: {v:+.4f}")


NameError: name 'gsm8k' is not defined

## 5. Shared Visualization Helpers

In [ ]:
def add_type_separator(ax, n_thinking=2, orientation='v', label=True):
    """Add a dashed line between thinking and instruct models."""
    pos = n_thinking - 0.5
    if orientation == 'v':
        ax.axvline(pos, color='#aaa', linestyle=':', linewidth=1.5, alpha=0.8)
        if label:
            ax.text(pos/2, ax.get_ylim()[1]*0.99, '← Thinking',
                    ha='center', va='top', fontsize=8, color='#555', style='italic')
            ax.text((pos + len(MODEL_ORDER))/2, ax.get_ylim()[1]*0.99, 'Instruct →',
                    ha='center', va='top', fontsize=8, color='#555', style='italic')
    else:
        ax.axhline(pos, color='#aaa', linestyle=':', linewidth=1.5, alpha=0.8)

def annotate_bars(ax, bars, fmt='{:.3f}', rotation=90, fontsize=7, pad=0.001):
    for bar in bars:
        v = bar.get_height()
        if np.isnan(v): continue
        ax.text(bar.get_x()+bar.get_width()/2, v+pad,
                fmt.format(v), ha='center', va='bottom',
                fontsize=fontsize, rotation=rotation, color='#333')

def annotate_hbars(ax, bars, fmt='{:+.3f}', fontsize=9, pad=0.001):
    for bar in bars:
        v = bar.get_width()
        if np.isnan(v): continue
        ha = 'left' if v >= 0 else 'right'
        ax.text(v + (pad if v >= 0 else -pad),
                bar.get_y()+bar.get_height()/2,
                fmt.format(v), va='center', ha=ha, fontsize=fontsize, color='#333')

print("✅ Helpers ready")


## Figure 1 — Accuracy per Persona Condition (GSM8K)
Grouped bar chart: 6 models × 5 conditions (Man-Exp, Man-Imp-Avg, W-Exp, W-Imp-Avg, Neutral)  
*Hypothesis: gender conditioning should not affect mathematical reasoning accuracy.*

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
x  = np.arange(len(MODEL_ORDER))
w  = 0.15
offs = [-2*w, -w, 0, w, 2*w]

conditions = [
    ('Man Explicit',       C['man_exp'], lambda d: d.get('man_explicit_1',   {}).get('accuracy', np.nan)),
    ('Man Implicit (Avg)', C['man_imp'], lambda d: avg_acc(d, PERSONAS_MAN)),
    ('Woman Explicit',     C['w_exp'],   lambda d: d.get('woman_explicit_1', {}).get('accuracy', np.nan)),
    ('Woman Implicit(Avg)',C['w_imp'],   lambda d: avg_acc(d, PERSONAS_WMAN)),
    ('Neutral',            C['neutral'], lambda d: d.get('neutral',          {}).get('accuracy', np.nan)),
]

for (label, color, fn), off in zip(conditions, offs):
    vals = [fn(gsm8k[m]) if m in gsm8k else np.nan for m in MODEL_ORDER]
    bars = ax.bar(x + off, vals, w*0.92, label=label, color=color, edgecolor='white', linewidth=0.5)
    annotate_bars(ax, bars, fmt='{:.3f}')

global_neut = np.nanmean([gsm8k[m]['neutral']['accuracy']
                           for m in MODEL_ORDER if m in gsm8k and 'neutral' in gsm8k[m]])
ax.axhline(global_neut, color='black', linestyle='--', linewidth=1.2, alpha=0.6,
           label=f'Global Neutral Avg ({global_neut:.3f})')

ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, fontsize=10)
ax.set_ylim(0.78, 1.02); ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Figure 1. Accuracy per Persona Condition — Gender Domain, GSM8K\n'
             '(6 Main Models  |  n = 1,319 questions per condition)', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9, ncol=2)
add_type_separator(ax, label=True)

plt.tight_layout()
savefig('Fig1_Accuracy_GSM8K')


## Figure 2 — Accuracy per Persona Condition (ETHICS)
Same structure as Fig 1 but on moral reasoning benchmark.  
*ETHICS is hypothesized to be more susceptible to persona framing than mathematical reasoning.*

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for (label, color, fn), off in zip(conditions, offs):
    vals = [fn(ethics[m]) if m in ethics else np.nan for m in MODEL_ORDER]
    bars = ax.bar(x + off, vals, w*0.92, label=label, color=color, edgecolor='white', linewidth=0.5)
    annotate_bars(ax, bars, fmt='{:.3f}')

global_neut_eth = np.nanmean([ethics[m]['neutral']['accuracy']
                               for m in MODEL_ORDER if m in ethics and 'neutral' in ethics[m]])
ax.axhline(global_neut_eth, color='black', linestyle='--', linewidth=1.2, alpha=0.6,
           label=f'Global Neutral Avg ({global_neut_eth:.3f})')

ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, fontsize=10)
ax.set_ylim(0.35, 0.58); ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Figure 2. Accuracy per Persona Condition — Gender Domain, ETHICS\n'
             '(6 Main Models  |  n = 1,514 questions per condition)', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9, ncol=2)
add_type_separator(ax, label=True)

plt.tight_layout()
savefig('Fig2_Accuracy_ETHICS')


## Figure 3 — P-Value Heatmap (Two-Proportion Z-Test)
★ p < .05  |  ★★ p < .01  |  ★★★ p < .001  
*Redder = more significant. White = not significant.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

for ax, pv, title in zip(axes, [gsm_pv, eth_pv], ['GSM8K', 'ETHICS']):
    mat = np.array([[pv.get(m, {}).get(c, np.nan) for c in COMP_LABELS] for m in MODEL_ORDER])

    annot = np.full(mat.shape, '', dtype=object)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if np.isnan(v):       annot[i,j] = 'N/A'
            elif v < 0.001:       annot[i,j] = f'p<.001\n★★★'
            elif v < 0.01:        annot[i,j] = f'p={v:.3f}\n★★'
            elif v < 0.05:        annot[i,j] = f'p={v:.3f}\n★'
            else:                 annot[i,j] = f'{v:.3f}'

    im = ax.imshow(mat, cmap='RdYlGn_r', vmin=0, vmax=0.12, aspect='auto')
    for i in range(len(MODEL_ORDER)):
        for j in range(len(COMP_LABELS)):
            v = mat[i, j]
            col = 'white' if (not np.isnan(v) and v < 0.05) else '#333'
            ax.text(j, i, annot[i,j], ha='center', va='center', fontsize=8.5, color=col)

    ax.set_xticks(range(len(COMP_LABELS)))
    ax.set_xticklabels(COMP_LABELS, rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(len(MODEL_ORDER)))
    ax.set_yticklabels(MODEL_ORDER, fontsize=9.5)
    ax.set_title(f'P-Value Heatmap — {title}', fontsize=11, fontweight='bold')
    ax.axhline(1.5, color='white', linewidth=2.5)

plt.colorbar(axes[1].images[0], ax=axes[1], shrink=0.85, label='p-value')
fig.suptitle('Figure 3. Two-Proportion Z-Test P-Value — Gender Domain\n'
             '(Aggregate Comparisons  |  6 Main Models)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
savefig('Fig3_Pvalue_Heatmap')


## Figure 4 — Cohen's h Effect Size Heatmap
Blue = Group A outperforms Group B (positive h).  Red = Group B outperforms Group A (negative h).  
★ marks statistically significant cells (p < .05).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

for ax, pv, hv, title in zip(axes, [gsm_pv, eth_pv], [gsm_h, eth_h], ['GSM8K', 'ETHICS']):
    mat_h = np.array([[hv.get(m, {}).get(c, np.nan) for c in COMP_LABELS] for m in MODEL_ORDER])
    mat_p = np.array([[pv.get(m, {}).get(c, np.nan) for c in COMP_LABELS] for m in MODEL_ORDER])

    im = ax.imshow(mat_h, cmap='RdBu', vmin=-0.15, vmax=0.15, aspect='auto')

    for i in range(len(MODEL_ORDER)):
        for j in range(len(COMP_LABELS)):
            h_val = mat_h[i,j]; p_val = mat_p[i,j]
            if np.isnan(h_val): continue
            star = '★' if (not np.isnan(p_val) and p_val < 0.05) else ''
            txt  = f'h={h_val:+.3f}{star}'
            col  = 'white' if abs(h_val) > 0.07 else '#333'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8.5, color=col)

    ax.set_xticks(range(len(COMP_LABELS)))
    ax.set_xticklabels(COMP_LABELS, rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(len(MODEL_ORDER)))
    ax.set_yticklabels(MODEL_ORDER, fontsize=9.5)
    ax.set_title(f"Cohen's h — {title}\n(blue = pos, red = neg, ★ = p<.05)", fontsize=11, fontweight='bold')
    ax.axhline(1.5, color='white', linewidth=2.5)

plt.colorbar(axes[1].images[0], ax=axes[1], shrink=0.85, label="Cohen's h")
fig.suptitle("Figure 4. Cohen's h Effect Size — Gender Domain\n"
             "(Aggregate Comparisons  |  6 Main Models)", fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
savefig('Fig4_CohenH_Heatmap')


## Figure 5 — Delta Analysis: Persona vs Neutral Baseline
Shows how much each gender persona condition shifts accuracy from the no-persona baseline.  
Positive = persona helps; Negative = persona suppresses.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, delta, title in zip(axes, [gsm_delta, eth_delta], ['GSM8K', 'ETHICS']):
    models = [m for m in MODEL_ORDER if m in delta]
    y = np.arange(len(models)); h = 0.35

    dm = [delta[m]['delta_man_neutral']   for m in models]
    dw = [delta[m]['delta_woman_neutral'] for m in models]

    bm = ax.barh(y+h/2, dm, h, color=C['man_exp'], alpha=0.85, label='Man Avg vs Neutral')
    bw = ax.barh(y-h/2, dw, h, color=C['w_exp'],   alpha=0.85, label='Woman Avg vs Neutral')
    annotate_hbars(ax, bm); annotate_hbars(ax, bw)

    ax.axvline(0,     color='black', linewidth=1)
    ax.fill_betweenx([-0.5, len(models)-0.5], -0.01, 0.01,
                     alpha=0.08, color='grey', label='±0.01 threshold')
    ax.set_yticks(y); ax.set_yticklabels(models, fontsize=10)
    ax.set_xlabel('Δ Accuracy (vs Neutral)', fontsize=10)
    ax.set_title(f'Delta vs Neutral — {title}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.axhline(1.5, color='#ccc', linewidth=1, linestyle='--')

fig.suptitle('Figure 5. Delta Analysis (Persona vs Neutral Baseline) — Gender Domain\n'
             '(Positive = persona improves accuracy over baseline)', fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('Fig5_Delta_Neutral')


## Figure 6 — Delta Analysis: Implicit vs Explicit Framing
Shows whether implicit persona formulation produces different accuracy than explicit.  
This is the key framing-effect analysis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, delta, title in zip(axes, [gsm_delta, eth_delta], ['GSM8K', 'ETHICS']):
    models = [m for m in MODEL_ORDER if m in delta]
    y = np.arange(len(models)); h = 0.35

    dm = [delta[m]['delta_man_ei']   for m in models]
    dw = [delta[m]['delta_woman_ei'] for m in models]

    bm = ax.barh(y+h/2, dm, h, color=C['man_imp'], alpha=0.85, label='Man: Implicit − Explicit')
    bw = ax.barh(y-h/2, dw, h, color=C['w_imp'],   alpha=0.85, label='Woman: Implicit − Explicit')
    annotate_hbars(ax, bm); annotate_hbars(ax, bw)

    ax.axvline(0, color='black', linewidth=1)
    ax.fill_betweenx([-0.5, len(models)-0.5], -0.01, 0.01,
                     alpha=0.08, color='grey', label='±0.01 threshold')
    ax.set_yticks(y); ax.set_yticklabels(models, fontsize=10)
    ax.set_xlabel('Δ Accuracy (Implicit − Explicit)', fontsize=10)
    ax.set_title(f'Delta Implicit vs Explicit — {title}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.axhline(1.5, color='#ccc', linewidth=1, linestyle='--')

fig.suptitle('Figure 6. Delta Analysis (Implicit vs Explicit Framing) — Gender Domain\n'
             '(Positive = implicit outperforms explicit)', fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('Fig6_Delta_ImpExp')


## Figure 7 — Token Usage per Model per Persona (GSM8K)
Shows average **prompt tokens** and **completion tokens** per question for each persona condition.  
Key hypothesis: Implicit prompts should produce significantly more prompt tokens (warm-up history in context).  
Thinking models (DeepSeek R1, Qwen3-Think) are expected to produce more completion tokens due to chain-of-thought reasoning.

In [ ]:
# ── Fig 7a: Avg Prompt Tokens per Persona ──────────────────────────────────
persona_display = {
    'man_explicit_1':   'M-Exp',
    'man_implicit_1':   'M-Imp1',
    'man_implicit_2':   'M-Imp2',
    'man_implicit_3':   'M-Imp3',
    'woman_explicit_1': 'W-Exp',
    'woman_implicit_1': 'W-Imp1',
    'woman_implicit_2': 'W-Imp2',
    'woman_implicit_3': 'W-Imp3',
    'neutral':          'Neutral',
}
persona_order_short = list(persona_display.keys())
persona_labels      = [persona_display[p] for p in persona_order_short]

# Color by persona type
persona_color = {
    'man_explicit_1': C['man_exp'],
    'man_implicit_1': C['man_imp'], 'man_implicit_2': C['man_imp'], 'man_implicit_3': C['man_imp'],
    'woman_explicit_1': C['w_exp'],
    'woman_implicit_1': C['w_imp'], 'woman_implicit_2': C['w_imp'], 'woman_implicit_3': C['w_imp'],
    'neutral': C['neutral'],
}

models_with_tokens = [m for m in MODEL_ORDER
                      if m in gsm8k and
                      any(gsm8k[m].get(p, {}).get('avg_prompt_tokens') for p in persona_order_short)]

n_m = len(models_with_tokens)
fig, axes = plt.subplots(1, n_m, figsize=(3.2*n_m, 5), sharey=False)
if n_m == 1: axes = [axes]

for ax, m in zip(axes, models_with_tokens):
    d = gsm8k[m]
    vals  = [d.get(p, {}).get('avg_prompt_tokens',   np.nan) for p in persona_order_short]
    colors= [persona_color[p] for p in persona_order_short]
    xs    = np.arange(len(persona_order_short))

    bars = ax.bar(xs, vals, color=colors, edgecolor='white', linewidth=0.4, width=0.75)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                    f'{int(v)}', ha='center', va='bottom', fontsize=7, rotation=90)

    ax.set_xticks(xs); ax.set_xticklabels(persona_labels, rotation=55, ha='right', fontsize=8)
    ax.set_title(m, fontsize=9, fontweight='bold')
    ax.set_ylabel('Avg Prompt Tokens / Question' if ax == axes[0] else '')

fig.suptitle('Figure 7a. Average Prompt Tokens per Persona — Gender GSM8K\n'
             '(Implicit prompts carry warm-up history → higher token count)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('Fig7a_PromptTokens')


In [ ]:
# ── Fig 7b: Avg Completion Tokens per Persona ───────────────────────────────
fig, axes = plt.subplots(1, n_m, figsize=(3.2*n_m, 5), sharey=False)
if n_m == 1: axes = [axes]

for ax, m in zip(axes, models_with_tokens):
    d     = gsm8k[m]
    vals  = [d.get(p, {}).get('avg_completion_tokens', np.nan) for p in persona_order_short]
    colors= [persona_color[p] for p in persona_order_short]
    xs    = np.arange(len(persona_order_short))

    bars = ax.bar(xs, vals, color=colors, edgecolor='white', linewidth=0.4, width=0.75)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                    f'{int(v)}', ha='center', va='bottom', fontsize=7, rotation=90)

    ax.set_xticks(xs); ax.set_xticklabels(persona_labels, rotation=55, ha='right', fontsize=8)
    ax.set_title(m, fontsize=9, fontweight='bold',
                 color=C['thinking'] if MODEL_TYPE.get(m)=='Thinking' else '#333')
    ax.set_ylabel('Avg Completion Tokens / Question' if ax == axes[0] else '')

fig.suptitle('Figure 7b. Average Completion Tokens per Persona — Gender GSM8K\n'
             '(Thinking models expected to produce significantly more completion tokens)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('Fig7b_CompletionTokens')


In [ ]:
# ── Fig 7c: Token ratio heatmap — 6 models × 9 personas ─────────────────────
# Rows = models, Cols = personas
# Value = avg_completion_tokens / avg_prompt_tokens (ratio)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

for ax, tok_type, label in zip(axes,
    ['avg_prompt_tokens', 'avg_completion_tokens'],
    ['Avg Prompt Tokens', 'Avg Completion Tokens']):

    mat = np.array([
        [gsm8k.get(m, {}).get(p, {}).get(tok_type, np.nan) for p in persona_order_short]
        for m in MODEL_ORDER
    ])

    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto',
                   vmin=np.nanmin(mat)*0.9, vmax=np.nanmax(mat)*1.05)

    for i in range(len(MODEL_ORDER)):
        for j in range(len(persona_order_short)):
            v = mat[i, j]
            if np.isnan(v): continue
            nv = (v - np.nanmin(mat)) / (np.nanmax(mat) - np.nanmin(mat) + 1e-9)
            col = 'white' if nv > 0.6 else '#333'
            ax.text(j, i, f'{int(v)}', ha='center', va='center', fontsize=9, color=col, fontweight='bold')

    ax.set_xticks(range(len(persona_order_short)))
    ax.set_xticklabels(persona_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(MODEL_ORDER)))
    ax.set_yticklabels(MODEL_ORDER, fontsize=9.5)
    ax.set_title(f'{label} per Persona', fontsize=11, fontweight='bold')
    ax.axhline(1.5, color='white', linewidth=2.5)
    plt.colorbar(im, ax=ax, shrink=0.85)

fig.suptitle('Figure 7c. Token Usage Heatmap — Gender GSM8K\n'
             '(Top separator: Thinking | Instruct)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
savefig('Fig7c_TokenHeatmap')


In [ ]:
# ── Fig 7d: Total tokens by model — stacked bar (prompt vs completion) ───────
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(MODEL_ORDER))
w = 0.35

for xi, m in enumerate(MODEL_ORDER):
    if m not in gsm8k: continue
    d = gsm8k[m]
    # Average across all persona conditions
    avg_p = np.nanmean([d.get(p,{}).get('avg_prompt_tokens', np.nan) for p in persona_order_short])
    avg_c = np.nanmean([d.get(p,{}).get('avg_completion_tokens', np.nan) for p in persona_order_short])

    b1 = ax.bar(xi - w/2, avg_p, w, color='#74ADD1', label='Avg Prompt' if xi==0 else '')
    b2 = ax.bar(xi + w/2, avg_c, w, color='#D6604D', label='Avg Completion' if xi==0 else '')

    for b, v in [(b1, avg_p), (b2, avg_c)]:
        if not np.isnan(v):
            ax.text(b[0].get_x()+b[0].get_width()/2, v+5,
                    f'{int(v)}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, fontsize=10)
ax.set_ylabel('Average Tokens per Question', fontsize=11)
ax.set_title('Figure 7d. Average Token Usage per Model (across all persona conditions) — GSM8K\n'
             '(Completion tokens for thinking models include chain-of-thought reasoning)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.axvline(1.5, color='#aaa', linestyle=':', linewidth=1.5)
ax.text(0.75, ax.get_ylim()[1]*0.97, '← Thinking', ha='center', fontsize=8, color='#555', style='italic')
ax.text(3.5,  ax.get_ylim()[1]*0.97, 'Instruct →',  ha='center', fontsize=8, color='#555', style='italic')

plt.tight_layout()
savefig('Fig7d_TokenByModel')


## Figure 8 — Linguistic Marker Heatmap (GSM8K)
8 key markers per model. Values = absolute density (per 100 tokens) or average word count.  
Data source: aggregate analysis document (pre-computed). Update `MARKER_DATA` if you have per-persona breakdown.

In [ ]:
# ── Edit MARKER_DATA if you have updated values ─────────────────────────────
MARKER_DATA = {
    'DeepSeek R1 32B': {'Sequential':0.83, 'Bullet':3.29, 'Sect.Header':4.36,
                        'Hedging':0.016, 'Self-Corr.':0.010, 'LaTeX':3.09, 'Bold':6.49, 'Word Count':105},
    'Qwen3-Think 80B': {'Sequential':0.97, 'Bullet':1.60, 'Sect.Header':2.09,
                        'Hedging':0.135, 'Self-Corr.':0.084, 'LaTeX':5.70, 'Bold':6.57, 'Word Count':201},
    'LLaMA 3.3 70B':   {'Sequential':2.29, 'Bullet':0.42, 'Sect.Header':0.04,
                        'Hedging':0.018, 'Self-Corr.':0.005, 'LaTeX':0.01, 'Bold':0.19, 'Word Count':130},
    'GPT-OSS 20B':     {'Sequential':0.63, 'Bullet':1.50, 'Sect.Header':0.79,
                        'Hedging':0.012, 'Self-Corr.':0.020, 'LaTeX':4.78, 'Bold':8.37, 'Word Count': 48},
    'LLaMA 3.1 8B':    {'Sequential':2.37, 'Bullet':0.50, 'Sect.Header':0.06,
                        'Hedging':0.017, 'Self-Corr.':0.008, 'LaTeX':0.00, 'Bold':0.09, 'Word Count':132},
    'Ministral 8B':    {'Sequential':1.00, 'Bullet':2.30, 'Sect.Header':3.33,
                        'Hedging':0.027, 'Self-Corr.':0.012, 'LaTeX':2.74, 'Bold':7.41, 'Word Count':151},
}
marker_names = list(next(iter(MARKER_DATA.values())).keys())

mat = np.array([[MARKER_DATA[m][mk] for mk in marker_names] for m in MODEL_ORDER], dtype=float)
# Column-normalize for color; show actual values as text
mat_norm = (mat - mat.min(0)) / (mat.max(0) - mat.min(0) + 1e-9)

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(mat_norm.T, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

for j, mk in enumerate(marker_names):
    for i, m in enumerate(MODEL_ORDER):
        v  = mat[i, j]
        nv = mat_norm[i, j]
        fmt = f'{v:.3f}' if v < 1 else (f'{v:.2f}' if v < 10 else f'{int(v)}')
        col = 'white' if nv > 0.62 else '#333'
        ax.text(i, j, fmt, ha='center', va='center', fontsize=9, color=col, fontweight='bold')

ax.set_xticks(range(len(MODEL_ORDER))); ax.set_xticklabels(MODEL_ORDER, fontsize=10)
ax.set_yticks(range(len(marker_names))); ax.set_yticklabels(marker_names, fontsize=9.5)
ax.set_title('Figure 8. Linguistic Marker Profile — Gender GSM8K\n'
             '(Color = normalized per marker; values = absolute density or avg word count)',
             fontsize=12, fontweight='bold')
ax.axvline(1.5, color='white', linewidth=2.5)

# Type indicator
for i, m in enumerate(MODEL_ORDER):
    t = MODEL_TYPE[m]
    ax.text(i, -0.8, '◆ Thinking' if t=='Thinking' else '○ Instruct',
            ha='center', fontsize=7.5,
            color=C['thinking'] if t=='Thinking' else '#888')

plt.colorbar(im, ax=ax, shrink=0.8, label='Normalized intensity')
plt.tight_layout()
savefig('Fig8_LinguisticMarker')


## Figure 9 — Summary: Gender Gap & EI Gap per Model
Dual-axis chart combining:  
- **Gender Gap** = |Man avg − Woman avg| (bar, left axis)  
- **EI Gap** = Implicit avg − Explicit avg (line, right axis)  
Useful for Bab III (model selection rationale) and Bab VI cross-domain analysis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, delta, title in zip(axes, [gsm_delta, eth_delta], ['GSM8K', 'ETHICS']):
    models = [m for m in MODEL_ORDER if m in delta]
    xi     = np.arange(len(models))

    gg = [delta[m]['gender_gap'] for m in models]
    ei = [delta[m]['ei_gap']     for m in models]

    ax2 = ax.twinx()
    bars  = ax.bar(xi - 0.15, gg, 0.3, color='#D6604D', alpha=0.82, label='Gender Gap |M−W|')
    line, = ax2.plot(xi + 0.15, ei, 'o--', color='#2166AC', markersize=9,
                     linewidth=2.2, label='EI Gap (Imp−Exp)')

    for bar, v in zip(bars, gg):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0002,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8, color='#D6604D', fontweight='bold')
    for xi_, v in zip(xi+0.15, ei):
        ax2.text(xi_, v + 0.0008, f'{v:+.4f}',
                 ha='center', va='bottom', fontsize=8, color='#2166AC', fontweight='bold')

    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, fontsize=9, rotation=20, ha='right')
    ax.set_ylabel('Gender Gap (|accuracy diff|)', color='#D6604D', fontsize=10)
    ax2.set_ylabel('EI Gap (Implicit − Explicit)', color='#2166AC', fontsize=10)
    ax.set_title(f'Gender Gap & EI Gap — {title}', fontsize=11, fontweight='bold')
    ax.axhline(0.01, color='#D6604D', linestyle=':', linewidth=0.8, alpha=0.5)
    ax2.axhline(0,   color='#2166AC', linestyle=':', linewidth=0.8, alpha=0.4)
    ax.axvline(1.5,  color='#aaa',    linestyle=':',  linewidth=1.2)

    l1, lb1 = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lb1+lb2, fontsize=8, loc='upper right')

fig.suptitle('Figure 9. Gender Gap and EI Gap Summary — Gender Domain\n'
             '(Gender Gap = |Man avg − Woman avg|  |  EI Gap = Implicit avg − Explicit avg)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('Fig9_GenderGap_EIGap')


## Figure 10 — Combined Significance Overview (GSM8K + ETHICS)
Quick visual showing which models × comparisons are significant across both benchmarks.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, pv, title in zip(axes, [gsm_pv, eth_pv], ['GSM8K', 'ETHICS']):
    mat = np.array([[pv.get(m, {}).get(c, np.nan) for c in COMP_LABELS] for m in MODEL_ORDER])
    sig = (mat < 0.05).astype(float)
    sig[np.isnan(mat)] = np.nan

    im = ax.imshow(sig, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)

    for i in range(len(MODEL_ORDER)):
        for j in range(len(COMP_LABELS)):
            v = mat[i, j]
            if np.isnan(v): continue
            mark = '★ SIG' if v < 0.05 else '—'
            col  = 'white' if v < 0.05 else '#aaa'
            pstr = f'p={v:.3f}\n{mark}' if v < 0.1 else f'{v:.3f}'
            ax.text(j, i, pstr, ha='center', va='center', fontsize=8, color=col)

    ax.set_yticks(range(len(MODEL_ORDER))); ax.set_yticklabels(MODEL_ORDER, fontsize=9.5)
    ax.set_title(f'{title} — Significance Map', fontsize=11, fontweight='bold', loc='left')
    ax.axhline(1.5, color='white', linewidth=2.5)

axes[1].set_xticks(range(len(COMP_LABELS)))
axes[1].set_xticklabels(COMP_LABELS, rotation=30, ha='right', fontsize=9)

fig.suptitle('Figure 10. Combined Significance Overview — Gender Domain\n'
             '(Green = not significant  |  Red = significant at p < .05)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('Fig10_SignificanceOverview')


---
## Notes & How to Extend

### To add Religion or Regional domain:
1. Load additional sheets: `ws_rel_gsm8k = sh.worksheet('Religion Persona GSM8K')`
2. Adjust `MODEL_MAP_*` if model names differ
3. Re-run the stat computation cells with the new dataset

### To update Linguistic Marker data:
- Edit the `MARKER_DATA` dictionary in Figure 8 cell directly

### To change figure style:
- Edit `plt.rcParams` in the Config section
- Edit `C` color dictionary

### Token data notes:
- `avg_prompt_tokens` for **implicit** conditions includes the warm-up conversation history
- `avg_completion_tokens` for **thinking models** includes chain-of-thought reasoning tokens
- If your sheet doesn't have token columns, Figs 7a–7d will show no data (gracefully skipped)
